# CredResolve — Operational Driver Investigation

**Purpose:** cover the assignment's required operational dimensions—channel, telephony vendor, calling time, attempt frequency, campaigns, agent/tenure, borrower geography/segment, and available dimensions—without confusing descriptive associations with causal effects.

**Important scope rules**
- Raw operational logs are imperfect. Calls and WhatsApp events have repeated IDs; repeated `call_id` groups include conflicting records. Ambiguous IDs are excluded from canonicalized outcome tables and counted for review.
- `event_at` timestamps are naive strings accompanied by source timezone fields. The normalized IST view below assumes each timestamp is a local wall-clock time in its row's `timezone`; confirm that assumption with source owners before production use.
- The data ends on **8 August 2026** for payments. Current account attributes are not historical as-of attributes.
- These comparisons are descriptive only. A touch before a payment is not proof the touch caused the payment. Agent identity, campaign-window, timezone and entity-resolution issues prevent a reliable causal ranking.
- The dataset has no explicit client or language field; those effects cannot be evaluated from the supplied columns.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Find project root without a personal-machine path.
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "dataset" / "accounts.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find dataset/accounts.csv. Save this notebook under CredResolve/notebooks "
        "and keep the original CSVs in CredResolve/dataset."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "dataset"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

files = [
    "accounts", "borrowers", "agents", "agent_sessions", "campaigns", "daily_targeting",
    "calls", "call_attempts", "vendor_telephony", "sms_events", "whatsapp_events",
    "field_visits", "promises_to_pay", "payments", "complaints"
]
data = {name: pd.read_csv(DATA_DIR / f"{name}.csv", low_memory=False) for name in files}
print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded {len(data)} source tables")
print(f"Payment event window: {pd.to_datetime(data['payments']['event_at'], errors='coerce').min()} to {pd.to_datetime(data['payments']['event_at'], errors='coerce').max()}")

Project root: C:\Users\GOPICHAND\Desktop\CredResolve
Loaded 15 source tables
Payment event window: 2026-01-01 00:14:40 to 2026-08-08 23:50:23


## 1. Which requested dimensions can the data actually support?

This matrix makes the limitations explicit before interpreting segment differences.

In [2]:
all_cols = {str(c).lower() for frame in data.values() for c in frame.columns}
client_cols = sorted(c for c in all_cols if "client" in c)
language_cols = sorted(c for c in all_cols if "language" in c or "lang_" in c)

coverage = pd.DataFrame([
    ("Portfolio mix / DPD / borrower segment", "accounts: loan_type, dpd, risk_segment; account-level payment summary", "Available; account attributes are current snapshot only", "Descriptive only; not causal"),
    ("Client", ", ".join(client_cols) if client_cols else "No client column found", "Unavailable in supplied schema", "Cannot test"),
    ("Geography", "borrowers: city/state; field_visits: latitude/longitude", "Fields exist, but borrower IDs have conflicting city/state values", "Quality diagnostic only until entity resolution"),
    ("Language", ", ".join(language_cols) if language_cols else "No language column found", "Unavailable in supplied schema", "Cannot test"),
    ("Agent / tenure", "agents, calls, call_attempts, agent_sessions", "Agent IDs conflict on employee code, name, vendor and joined date", "Do not rank agent/tenure performance"),
    ("Campaign", "campaigns + daily_targeting", "Targeting dates often fall outside campaign active calendar dates", "Attribution requires reconciliation"),
    ("Channel", "calls, campaigns, agent_sessions, SMS and WhatsApp events, field_visits", "Available as event labels; event definitions differ across tables", "Descriptive event mix only"),
    ("Telephony vendor", "calls.vendor_id + vendor_telephony", "Available; duplicate call IDs and timezone mapping need review", "Descriptive vendor comparison only"),
    ("Calling time", "calls.event_at + calls.timezone", "Available, but timezone interpretation is assumed in the IST normalization below", "Exploratory; validate source time semantics"),
    ("Attempt frequency", "call_attempts.attempt_no and account_id", "Available; attempt outcome is not a recovery outcome", "Descriptive association only"),
], columns=["requested_dimension", "available_fields", "data_fitness", "allowed_interpretation"])
display(coverage)
coverage.to_csv(OUTPUT_DIR / "driver_dimension_coverage.csv", index=False)

,requested_dimension,available_fields,data_fitness,allowed_interpretation
0,Portfolio mix / DPD / borrower segment,"accounts: loan_type, dpd, risk_segment; accoun...",Available; account attributes are current snap...,Descriptive only; not causal
1,Client,No client column found,Unavailable in supplied schema,Cannot test
2,Geography,borrowers: city/state; field_visits: latitude/...,"Fields exist, but borrower IDs have conflictin...",Quality diagnostic only until entity resolution
3,Language,No language column found,Unavailable in supplied schema,Cannot test
4,Agent / tenure,"agents, calls, call_attempts, agent_sessions","Agent IDs conflict on employee code, name, ven...",Do not rank agent/tenure performance
5,Campaign,campaigns + daily_targeting,Targeting dates often fall outside campaign ac...,Attribution requires reconciliation
6,Channel,"calls, campaigns, agent_sessions, SMS and What...",Available as event labels; event definitions d...,Descriptive event mix only
7,Telephony vendor,calls.vendor_id + vendor_telephony,Available; duplicate call IDs and timezone map...,Descriptive vendor comparison only
8,Calling time,calls.event_at + calls.timezone,"Available, but timezone interpretation is assu...",Exploratory; validate source time semantics
9,Attempt frequency,call_attempts.attempt_no and account_id,Available; attempt outcome is not a recovery o...,Descriptive association only


## 2. Calls and telephony vendor

The source has 91,350 call rows and 90,000 distinct call IDs. The next cell examines repeated IDs across the fields that define a call. For vendor/status comparisons, exact or core-consistent duplicate IDs are represented once; IDs whose core fields conflict are excluded and counted. The output is still observational—not an estimate of vendor causality.

In [3]:
calls = data["calls"].copy()
call_core = ["account_id", "borrower_id", "event_at", "agent_id", "campaign_id", "direction", "vendor_id", "call_status", "duration_sec", "timezone"]
call_groups = calls.dropna(subset=["call_id"]).groupby("call_id")[call_core].nunique(dropna=False).max(axis=1)
call_counts = calls["call_id"].value_counts(dropna=True)
duplicate_call_groups = int((call_counts > 1).sum())
conflicting_call_ids = set(call_groups[call_groups > 1].index)
canonical_calls = calls[calls["call_id"].notna() & ~calls["call_id"].isin(conflicting_call_ids)].drop_duplicates("call_id", keep="first").copy()
call_quality = pd.DataFrame([
    ("Raw call rows", len(calls)),
    ("Distinct non-missing call IDs", calls["call_id"].nunique(dropna=True)),
    ("Repeated call-ID groups", duplicate_call_groups),
    ("Call IDs with conflicting core fields", len(conflicting_call_ids)),
    ("Rows represented in canonical call comparison", len(canonical_calls)),
], columns=["check", "value"])
display(call_quality)
call_quality.to_csv(OUTPUT_DIR / "call_identity_quality.csv", index=False)

vendor_map = data["vendor_telephony"][["vendor_id", "vendor_name", "status", "timezone", "schema_version"]].drop_duplicates("vendor_id")
canonical_calls["duration_sec"] = pd.to_numeric(canonical_calls["duration_sec"], errors="coerce")
canonical_calls["is_answered"] = canonical_calls["call_status"].astype(str).str.upper().eq("ANSWERED")
answered_durations = canonical_calls[canonical_calls["is_answered"]].groupby("vendor_id")["duration_sec"].median()
vendor_summary = canonical_calls.groupby("vendor_id").agg(
    canonical_calls=("call_id", "nunique"),
    answered_calls=("is_answered", "sum"),
    median_duration_all_sec=("duration_sec", "median"),
).reset_index()
vendor_summary["answered_share_pct"] = (100 * vendor_summary["answered_calls"] / vendor_summary["canonical_calls"]).round(2)
vendor_summary["median_answered_duration_sec"] = vendor_summary["vendor_id"].map(answered_durations).round(1)
vendor_summary = vendor_summary.merge(vendor_map, on="vendor_id", how="left", validate="many_to_one")
vendor_summary = vendor_summary.sort_values("canonical_calls", ascending=False)
display(vendor_summary)
vendor_summary.to_csv(OUTPUT_DIR / "calls_by_vendor_descriptive.csv", index=False)

,check,value
0,Raw call rows,91350
1,Distinct non-missing call IDs,90000
2,Repeated call-ID groups,1350
3,Call IDs with conflicting core fields,79
4,Rows represented in canonical call comparison,89921


,vendor_id,canonical_calls,answered_calls,median_duration_all_sec,answered_share_pct,median_answered_duration_sec,vendor_name,status,timezone,schema_version
9,VND0000010,6159,1251,454.0,20.31,443.0,Airtel,INACTIVE,UTC,v1
10,VND0000011,6066,1218,455.0,20.08,453.0,Airtel,ACTIVE,UTC,v3
6,VND0000007,6064,1179,441.0,19.44,432.0,TataTele,INACTIVE,UTC,v1
14,VND0000015,6061,1191,451.0,19.65,445.0,TataTele,ACTIVE,Asia/Kolkata,v2
4,VND0000005,6041,1177,457.0,19.48,473.0,Twilio,ACTIVE,Asia/Kolkata,v1
13,VND0000014,6036,1273,455.5,21.09,454.0,Knowlarity,INACTIVE,Asia/Kolkata,v3
0,VND0000001,6019,1167,451.0,19.39,445.0,Airtel,INACTIVE,Asia/Kolkata,v3
11,VND0000012,6005,1167,451.0,19.43,446.0,Airtel,INACTIVE,UTC,v1
8,VND0000009,5968,1152,455.0,19.30,481.0,Exotel,INACTIVE,Asia/Kolkata,v1
2,VND0000003,5962,1188,460.0,19.93,473.5,Twilio,INACTIVE,UTC,v3


## 3. Calling time (exploratory)

Each source timestamp is localized using its row-level `timezone` and converted to Asia/Kolkata (IST). **This relies on the assumption that `event_at` is a wall-clock timestamp in the stated timezone.** If timestamps were already normalized to UTC or the zone field reflects a different meaning, these buckets will be wrong; confirm with the data owner before operationalizing.

In [4]:
def local_wallclock_to_ist(frame, dt_col, tz_col, output_col="event_at_ist"):
    raw = pd.to_datetime(frame[dt_col], errors="coerce")
    result = pd.Series(pd.NaT, index=frame.index, dtype="datetime64[ns]")
    for tz, idx in frame.groupby(tz_col, dropna=False).groups.items():
        if pd.isna(tz) or not str(tz).strip():
            continue
        try:
            localized = raw.loc[idx].dt.tz_localize(str(tz), ambiguous="NaT", nonexistent="NaT")
            converted = localized.dt.tz_convert("Asia/Kolkata").dt.tz_localize(None)
            result.loc[idx] = converted
        except Exception:
            # Invalid/unrecognized timezone group remains missing and is quantified below.
            continue
    out = frame.copy()
    out[output_col] = result
    return out

calls_time = local_wallclock_to_ist(canonical_calls, "event_at", "timezone")
calls_time["ist_hour"] = calls_time["event_at_ist"].dt.hour
calls_time["ist_hour_band"] = pd.cut(
    calls_time["ist_hour"], bins=[-1, 5, 10, 15, 19, 23],
    labels=["00–05", "06–10", "11–15", "16–19", "20–23"]
)
time_summary = calls_time.groupby("ist_hour_band", observed=False).agg(
    calls=("call_id", "nunique"), answered_calls=("is_answered", "sum"),
    rows_with_normalized_time=("event_at_ist", "count")
).reset_index()
time_summary["answered_share_pct"] = (100 * time_summary["answered_calls"] / time_summary["calls"].replace(0, np.nan)).round(2)
print("Calls with valid normalized IST timestamp:", int(calls_time["event_at_ist"].notna().sum()), "of", len(calls_time))
display(time_summary)
time_summary.to_csv(OUTPUT_DIR / "calls_by_ist_hour_exploratory.csv", index=False)

Calls with valid normalized IST timestamp: 89921 of 89921


,ist_hour_band,calls,answered_calls,rows_with_normalized_time,answered_share_pct
0,00–05,22614,4493,22614,19.87
1,06–10,18673,3661,18673,19.61
2,11–15,18741,3771,18741,20.12
3,16–19,14986,2896,14986,19.32
4,20–23,14907,3043,14907,20.41


## 4. Attempt frequency and outcomes

`CONNECTED` is a call-attempt status—not proof of a right-party contact or payment. Attempts are not randomized: account difficulty and targeting rules can affect both attempt count and payment coverage. Therefore the tables below are descriptive associations only.

In [5]:
attempts = data["call_attempts"].copy()
attempts["attempt_no"] = pd.to_numeric(attempts["attempt_no"], errors="coerce")
attempt_outcomes = attempts.groupby("attempt_no", dropna=False).agg(
    attempt_rows=("attempt_id", "count"),
    distinct_attempt_ids=("attempt_id", "nunique"),
    connected_attempts=("attempt_status", lambda s: s.astype(str).str.upper().eq("CONNECTED").sum()),
    accounts_attempted=("account_id", "nunique")
).reset_index().sort_values("attempt_no")
attempt_outcomes["connected_share_pct"] = (100 * attempt_outcomes["connected_attempts"] / attempt_outcomes["attempt_rows"].replace(0, np.nan)).round(2)
display(attempt_outcomes)
attempt_outcomes.to_csv(OUTPUT_DIR / "attempt_outcomes_by_attempt_number.csv", index=False)

attempts_per_account = attempts.groupby("account_id")["attempt_id"].nunique().rename("attempt_count")
account_base = data["accounts"][["account_id", "risk_segment", "dpd", "loan_type"]].drop_duplicates("account_id")
payments = data["payments"].copy()
payments["amount"] = pd.to_numeric(payments["amount"], errors="coerce")
payments["event_at"] = pd.to_datetime(payments["event_at"], errors="coerce")
# Canonicalize payment IDs only where all core transaction fields agree.
pay_core = ["account_id", "borrower_id", "event_at", "amount", "payment_status", "payment_method", "provider_id"]
pay_ver = payments.dropna(subset=["payment_id"]).groupby("payment_id")[pay_core].nunique(dropna=False).max(axis=1)
pay_conflicts = set(pay_ver[pay_ver > 1].index)
pay_canonical = payments[payments["payment_id"].notna() & ~payments["payment_id"].isin(pay_conflicts)].drop_duplicates("payment_id", keep="first")
success = pay_canonical[pay_canonical["payment_status"].astype(str).str.upper().eq("SUCCESS")]
success_accounts = success.groupby("account_id").agg(successful_payment_ids=("payment_id", "nunique"), gross_successful_amount=("amount", "sum"))
account_base = account_base.merge(attempts_per_account, left_on="account_id", right_index=True, how="left")
account_base = account_base.merge(success_accounts, left_on="account_id", right_index=True, how="left")
account_base["attempt_count"] = account_base["attempt_count"].fillna(0).astype(int)
account_base["successful_payment_ids"] = account_base["successful_payment_ids"].fillna(0).astype(int)
account_base["gross_successful_amount"] = account_base["gross_successful_amount"].fillna(0.0)
account_base["has_success"] = account_base["successful_payment_ids"].gt(0)
account_base["attempt_band"] = pd.cut(account_base["attempt_count"], bins=[-1, 0, 1, 3, 5, np.inf], labels=["0", "1", "2–3", "4–5", "6+"])
frequency_summary = account_base.groupby("attempt_band", observed=False).agg(
    accounts=("account_id", "nunique"),
    accounts_with_success=("has_success", "sum"),
    gross_successful_amount=("gross_successful_amount", "sum"),
    median_attempt_count=("attempt_count", "median")
).reset_index()
frequency_summary["account_payment_coverage_pct"] = (100 * frequency_summary["accounts_with_success"] / frequency_summary["accounts"].replace(0, np.nan)).round(2)
frequency_summary["gross_successful_amount"] = frequency_summary["gross_successful_amount"].round(2)
display(frequency_summary)
frequency_summary.to_csv(OUTPUT_DIR / "attempt_frequency_vs_payment_coverage.csv", index=False)

,attempt_no,attempt_rows,distinct_attempt_ids,connected_attempts,accounts_attempted,connected_share_pct
0,1,17089,17089,3465,13037,20.28
1,2,17100,17100,3383,13030,19.78
2,3,17201,17201,3538,13171,20.57
3,4,17252,17252,3472,13075,20.13
4,5,16952,16952,3406,12987,20.09
5,6,17207,17207,3336,13112,19.39
6,7,17199,17199,3423,13103,19.90


,attempt_band,accounts,accounts_with_success,gross_successful_amount,median_attempt_count,account_payment_coverage_pct
0,0,549,240,2.349092e+07,0.0,43.72
1,1,2124,902,9.039270e+07,1.0,42.47
2,2–3,10377,4615,4.574136e+08,3.0,44.47
3,4–5,10553,4677,4.643581e+08,4.0,44.32
4,6+,6397,2850,2.799287e+08,7.0,44.55


## 5. Digital engagement and field operations

SMS/WhatsApp event types and field-visit outcomes are source-reported operational signals. Event names differ by channel and do not share one common denominator. We do not claim channel conversion or incremental recovery from these counts.

In [6]:
def event_quality(frame, id_col, core_fields):
    counts = frame[id_col].value_counts(dropna=True)
    groups = frame.dropna(subset=[id_col]).groupby(id_col)[core_fields].nunique(dropna=False).max(axis=1)
    conflict_ids = set(groups[groups > 1].index)
    canonical = frame[frame[id_col].notna() & ~frame[id_col].isin(conflict_ids)].drop_duplicates(id_col, keep="first").copy()
    return canonical, int((counts > 1).sum()), len(conflict_ids)

sms = data["sms_events"].copy()
sms_core = ["account_id", "borrower_id", "event_at", "message_id", "event_type", "template_code", "provider_id"]
sms_canonical, sms_dupe_groups, sms_conflicts = event_quality(sms, "sms_event_id", sms_core)
wa = data["whatsapp_events"].copy()
wa_core = ["account_id", "borrower_id", "event_at", "message_id", "event_type", "template_code", "provider_id"]
wa_canonical, wa_dupe_groups, wa_conflicts = event_quality(wa, "whatsapp_event_id", wa_core)

digital_parts=[]
for channel, frame, id_col, raw, dup_groups, conflict_groups in [
    ("SMS", sms_canonical, "sms_event_id", sms, sms_dupe_groups, sms_conflicts),
    ("WhatsApp", wa_canonical, "whatsapp_event_id", wa, wa_dupe_groups, wa_conflicts),
]:
    event_type = frame.groupby("event_type", dropna=False).agg(events=(id_col, "nunique"), unique_accounts=("account_id", "nunique")).reset_index()
    event_type["channel"] = channel
    event_type["raw_source_rows"] = len(raw)
    event_type["raw_duplicate_id_groups"] = dup_groups
    event_type["conflicting_id_groups_excluded"] = conflict_groups
    digital_parts.append(event_type)
digital_summary = pd.concat(digital_parts, ignore_index=True)[["channel", "event_type", "events", "unique_accounts", "raw_source_rows", "raw_duplicate_id_groups", "conflicting_id_groups_excluded"]]
display(digital_summary.sort_values(["channel", "event_type"]))
digital_summary.to_csv(OUTPUT_DIR / "digital_event_mix_descriptive.csv", index=False)

visits = data["field_visits"].copy()
visit_summary = visits.groupby(["visit_type", "outcome"], dropna=False).agg(
    visit_rows=("visit_id", "count"),
    distinct_visits=("visit_id", "nunique"),
    unique_accounts=("account_id", "nunique"),
    rows_with_coordinates=("latitude", lambda s: s.notna().sum())
).reset_index()
display(visit_summary)
visit_summary.to_csv(OUTPUT_DIR / "field_visit_outcomes_descriptive.csv", index=False)

visit_totals = visits.groupby("visit_type").agg(
    visits=("visit_id", "nunique"),
    accounts_visited=("account_id", "nunique"),
    paid_outcome_visits=("outcome", lambda s: s.astype(str).str.upper().eq("PAID").sum())
).reset_index()
visit_totals["reported_paid_outcome_share_pct"] = (100 * visit_totals["paid_outcome_visits"] / visit_totals["visits"].replace(0, np.nan)).round(2)
display(visit_totals)
visit_totals.to_csv(OUTPUT_DIR / "field_visit_paid_share_by_type.csv", index=False)

,channel,event_type,events,unique_accounts,raw_source_rows,raw_duplicate_id_groups,conflicting_id_groups_excluded
0,SMS,CLICKED,11139,9251,45000,0,0
1,SMS,DELIVERED,11219,9334,45000,0,0
2,SMS,FAILED,11251,9395,45000,0,0
3,SMS,SENT,11391,9503,45000,0,0
4,WhatsApp,DELIVERED,9907,8401,60600,600,0
5,WhatsApp,FAILED,10029,8518,60600,600,0
6,WhatsApp,PAYMENT_CLICK,10056,8505,60600,600,0
7,WhatsApp,READ,9928,8387,60600,600,0
8,WhatsApp,REPLIED,9935,8434,60600,600,0
9,WhatsApp,SENT,10145,8635,60600,600,0


,visit_type,outcome,visit_rows,distinct_visits,unique_accounts,rows_with_coordinates
0,AFTER_HOURS,CONTACTED,1065,1065,1042,1065
1,AFTER_HOURS,NOT_AVAILABLE,1042,1042,1029,1042
2,AFTER_HOURS,PAID,1012,1012,998,1012
3,AFTER_HOURS,PTP,1013,1013,996,1013
4,AFTER_HOURS,REFUSED,1033,1033,1018,1033
5,AFTER_HOURS,WRONG_ADDRESS,1027,1027,1015,1027
6,FIRST_VISIT,CONTACTED,1036,1036,1026,1036
7,FIRST_VISIT,NOT_AVAILABLE,1013,1013,989,1013
8,FIRST_VISIT,PAID,1048,1048,1028,1048
9,FIRST_VISIT,PTP,1062,1062,1037,1062


,visit_type,visits,accounts_visited,paid_outcome_visits,reported_paid_outcome_share_pct
0,AFTER_HOURS,6192,5621,1012,16.34
1,FIRST_VISIT,6275,5632,1048,16.70
2,FOLLOWUP,6271,5673,1004,16.01
3,NOTICE,6262,5663,1016,16.22


## 6. PTP maturity and campaign targeting

PTP status is reported by the source and is not reconciled to payment IDs. Future-due promises at the end of the observed payment period are right-censored and must not be automatically classified as broken. Campaign alignment below is a calendar-date diagnostic; it does not replace timezone-normalized attribution.

In [7]:
cutoff = pd.Timestamp("2026-08-08 23:59:59")
ptp = data["promises_to_pay"].copy()
ptp["promised_date_dt"] = pd.to_datetime(ptp["promised_date"], errors="coerce")
ptp["is_mature_by_cutoff"] = ptp["promised_date_dt"].le(cutoff)
ptp_summary = ptp.groupby(["is_mature_by_cutoff", "status"], dropna=False).agg(promises=("ptp_id", "count"), unique_accounts=("account_id", "nunique")).reset_index()
display(ptp_summary)
mat = ptp[ptp["is_mature_by_cutoff"] & ptp["status"].astype(str).str.upper().isin(["KEPT", "BROKEN"])]
kept = int(mat["status"].astype(str).str.upper().eq("KEPT").sum())
broken = int(mat["status"].astype(str).str.upper().eq("BROKEN").sum())
ptp_rate = (100 * kept / (kept + broken)) if (kept + broken) else np.nan
print(f"Status-reported PTP kept proxy among matured KEPT/BROKEN records: {ptp_rate:.2f}% ({kept}/{kept+broken})")
print("Interpretation caveat: excludes OPEN/CANCELLED, does not reconcile promise status to actual payment, and relies on promised_date as the maturity boundary.")
ptp_summary.to_csv(OUTPUT_DIR / "ptp_maturity_status.csv", index=False)

campaigns = data["campaigns"].copy()
targets = data["daily_targeting"].copy()
targets["target_date_dt"] = pd.to_datetime(targets["target_date"], errors="coerce").dt.normalize()
campaigns["start_date"] = pd.to_datetime(campaigns["start_at"], errors="coerce").dt.normalize()
campaigns["end_date"] = pd.to_datetime(campaigns["end_at"], errors="coerce").dt.normalize()
target_campaign = targets.merge(campaigns[["campaign_id", "channel", "strategy_version", "start_date", "end_date"]], on="campaign_id", how="left", validate="many_to_one")
target_campaign["window_status"] = np.select(
    [target_campaign["target_date_dt"].lt(target_campaign["start_date"]), target_campaign["target_date_dt"].gt(target_campaign["end_date"])],
    ["BEFORE_CAMPAIGN", "AFTER_CAMPAIGN"], default="WITHIN_CALENDAR_WINDOW"
)
campaign_summary = target_campaign.groupby(["channel", "strategy_version", "window_status"], dropna=False).agg(
    targeting_rows=("target_id", "count"), accounts=("account_id", "nunique")
).reset_index()
display(campaign_summary.sort_values(["channel", "strategy_version", "window_status"]))
campaign_summary.to_csv(OUTPUT_DIR / "campaign_targeting_window_by_channel_version.csv", index=False)

,is_mature_by_cutoff,status,promises,unique_accounts
0,False,BROKEN,333,329
1,False,CANCELLED,313,311
2,False,KEPT,309,309
3,False,OPEN,287,287
4,True,BROKEN,4220,3983
5,True,CANCELLED,4230,3952
6,True,KEPT,4180,3896
7,True,OPEN,4128,3850


Status-reported PTP kept proxy among matured KEPT/BROKEN records: 49.76% (4180/8400)
Interpretation caveat: excludes OPEN/CANCELLED, does not reconcile promise status to actual payment, and relies on promised_date as the maturity boundary.


,channel,strategy_version,window_status,targeting_rows,accounts
0,FIELD,legacy,AFTER_CAMPAIGN,1046,1024
1,FIELD,legacy,BEFORE_CAMPAIGN,727,719
2,FIELD,legacy,WITHIN_CALENDAR_WINDOW,494,486
3,FIELD,v1,AFTER_CAMPAIGN,713,705
4,FIELD,v1,BEFORE_CAMPAIGN,502,495
5,FIELD,v1,WITHIN_CALENDAR_WINDOW,328,325
6,FIELD,v2,AFTER_CAMPAIGN,543,541
7,FIELD,v2,BEFORE_CAMPAIGN,851,840
8,FIELD,v2,WITHIN_CALENDAR_WINDOW,453,447
9,FIELD,v3,AFTER_CAMPAIGN,710,704


## 7. Agent tenure and geography: quality gates before segment ranking

The sources contain agent tenure and borrower geography fields, but identity collisions are too widespread to support reliable ranking. The next outputs quantify the issue; they intentionally do not rank agents or geographies by recovery.

In [8]:
agents = data["agents"].copy()
agent_fields = ["employee_code", "agent_name", "vendor_id", "joined_at", "team"]
agent_versions = agents.groupby("agent_id")[agent_fields].nunique(dropna=False)
agent_quality_rows = [("Agent rows", len(agents)), ("Distinct agent IDs", agents["agent_id"].nunique())]
for field in agent_fields:
    agent_quality_rows.append((f"Agent IDs with multiple {field} values", int((agent_versions[field] > 1).sum())))
agent_quality = pd.DataFrame(agent_quality_rows, columns=["check", "value"])
display(agent_quality)
agent_quality.to_csv(OUTPUT_DIR / "agent_identity_tenure_quality.csv", index=False)

borrowers = data["borrowers"].copy()
borrower_fields = ["name", "phone", "email", "city", "state"]
borrower_versions = borrowers.groupby("borrower_id")[borrower_fields].nunique()
borrower_quality_rows = [("Borrower source rows", len(borrowers)), ("Distinct borrower IDs", borrowers["borrower_id"].nunique())]
for field in borrower_fields:
    borrower_quality_rows.append((f"Borrower IDs with multiple {field} values", int((borrower_versions[field] > 1).sum())))
account_borrower_ids = set(data["accounts"]["borrower_id"].dropna())
known_borrower_ids = set(borrowers["borrower_id"].dropna())
borrower_quality_rows.extend([
    ("Account rows missing borrower ID", int(data["accounts"]["borrower_id"].isna().sum())),
    ("Account rows with non-null borrower ID absent from borrower table", int((data["accounts"]["borrower_id"].notna() & ~data["accounts"]["borrower_id"].isin(known_borrower_ids)).sum())),
])
borrower_quality = pd.DataFrame(borrower_quality_rows, columns=["check", "value"])
display(borrower_quality)
borrower_quality.to_csv(OUTPUT_DIR / "borrower_geography_quality.csv", index=False)

visits["latitude"] = pd.to_numeric(visits["latitude"], errors="coerce")
visits["longitude"] = pd.to_numeric(visits["longitude"], errors="coerce")
coord_qc = pd.DataFrame([
    ("Field visit rows", len(visits)),
    ("Rows with both latitude and longitude", int((visits["latitude"].notna() & visits["longitude"].notna()).sum())),
    ("Distinct rounded coordinate pairs", int(visits.loc[visits["latitude"].notna() & visits["longitude"].notna(), ["latitude", "longitude"]].round(2).drop_duplicates().shape[0])),
], columns=["check", "value"])
display(coord_qc)
coord_qc.to_csv(OUTPUT_DIR / "field_visit_geography_quality.csv", index=False)

,check,value
0,Agent rows,30000
1,Distinct agent IDs,1000
2,Agent IDs with multiple employee_code values,1000
3,Agent IDs with multiple agent_name values,1000
4,Agent IDs with multiple vendor_id values,1000
5,Agent IDs with multiple joined_at values,1000
6,Agent IDs with multiple team values,1000


,check,value
0,Borrower source rows,30600
1,Distinct borrower IDs,11015
2,Borrower IDs with multiple name values,8185
3,Borrower IDs with multiple phone values,8399
4,Borrower IDs with multiple email values,8334
5,Borrower IDs with multiple city values,8159
6,Borrower IDs with multiple state values,8070
7,Account rows missing borrower ID,455
8,Account rows with non-null borrower ID absent ...,2458


,check,value
0,Field visit rows,25000
1,Rows with both latitude and longitude,25000
2,Distinct rounded coordinate pairs,24969


## 8. Conclusions mapped to the assignment

| Required dimension | Result from this investigation | Evidence classification / limitation |
|---|---|---|
| Channel | SMS, WhatsApp, call and field event mix can be described, but event schemas and denominators differ; no defensible common causal conversion rate. | Descriptive facts only |
| Telephony vendor | Source-reported answer share and duration can be summarized after call-ID quality checks; vendor timezone/schema/status history still needs validation. | Correlation only |
| Calling time | Exploratory time-of-day outcomes depend on the assumption that source timestamps are local to the supplied timezone. | Hypothesis until timestamps are confirmed |
| Attempt frequency | Connection share varies by attempt number; attempt-band account payment coverage is observational and confounded by targeting/borrower difficulty. | Correlation only |
| Campaign | Many targeting dates sit outside campaign active calendar dates; strategy labels do not identify one clean cutover date. | Strong evidence of attribution risk |
| Agent / tenure | Every distinct agent ID has multiple employee codes, names, vendors and joined dates in the source snapshot. | Data quality blocks reliable performance estimates |
| Geography | City/state fields are inconsistent for many borrower IDs; field-visit coordinates exist but need a stable geographic key and privacy-safe aggregation. | Data quality blocks reliable geography effects |
| Borrower segment / DPD / loan type | Step 4 reports modest differences in current-snapshot account payment coverage; these are not historical or causal effects. | Descriptive correlation only |
| Client / language | No explicit fields found in the supplied schema. | Not testable from supplied data |

**Decision implication:** this analysis does not establish which operational driver caused changes in gross payments. The next defensible move remains a controlled targeting pilot after identity, event-time, campaign and net-payment measurement are repaired. It does not justify spending the full ₹10 crore on an untested driver.

## 9. Saved outputs

CSV outputs are written to the project `outputs/` folder. Rerun this notebook after changing source files or cleaning decisions. The outputs are descriptive QA/driver tables and should not be interpreted as causal estimates.